<a href="https://colab.research.google.com/github/41340202s-jpg/ProgramingLanguage/blob/main/%E3%80%8CHW1_%E6%97%A5%E5%B8%B8%E6%94%AF%E5%87%BA%E9%80%9F%E7%AE%97%E8%88%87%E5%88%86%E6%94%A4_ipynb%E3%80%8D%E7%9A%84%E5%89%AF%E6%9C%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#日常支出速算與分攤（作業一）
- 目標：從 Sheet 讀「消費紀錄」→ 計總額/分類小計/AA 分攤 → 寫回 Sheet Summary 分頁。
- AI 點子（可選）：請模型總結本週花錢習慣與建議（例如「外食過多」）。
- Sheet 欄位：date, category, item, amount, payer

GoogleSheet: https://docs.google.com/spreadsheets/d/1H_Qe7YSPpavuUY-PM7Z3CZ9icmIYVVlQHQqYGvlE-b4/edit?usp=sharing

In [1]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
creds, _ = default()

gc = gspread.authorize(creds)

In [2]:
import pandas as pd
# read data and put it in a dataframe
# 在 google 工作表載入 gsheets
gsheets = gc.open_by_url('https://docs.google.com/spreadsheets/d/1H_Qe7YSPpavuUY-PM7Z3CZ9icmIYVVlQHQqYGvlE-b4/edit?usp=sharing')

In [3]:
# 從 gsheets 的 All-whiteboard-device 載入 sheets
sheets = gsheets.worksheet('工作表1').get_all_values()
# 將 sheets1 資料載入 pd 的 DataFrame 進行分析
df = pd.DataFrame(sheets[1:], columns=sheets[0])
# 取得最前面的5筆資料
df.head()

,日期,時間,分類,品項,金額,付款人,備註
0,2026-03-05,9:27,,早餐,50,,
1,2026-03-04,16:25,,麵包,50,ay,無
2,2026-03-03,15:04,,書,350,k,無


In [4]:
import datetime

In [5]:
# 讓使用者輸入資料
date_str = input("請輸入日期 (格式：YYYY-MM-DD): ")
# 檢查日期格式是否正確
datetime.datetime.strptime(date_str, '%Y-%m-%d')

time_str = input("請輸入時間 (格式：HH:MM): ")
# 檢查時間格式是否正確
datetime.datetime.strptime(time_str, '%H:%M')
cata = input("請輸入分類: ")
item = input("請輸入品項: ")
amount = float(input("請輸入金額: "))

請輸入日期 (格式：YYYY-MM-DD): 2026-02-15
請輸入時間 (格式：HH:MM): 15:57
請輸入品項: 食
請輸入品項: 零食
請輸入金額: 30


In [11]:
# 創建一個新的 DataFrame 來代表你要新增的資料
new_data = pd.DataFrame([{
    '日期': date_str,
    '時間': time_str,
    '分類': cata,
    '品項': item,
    '金額': amount
}])

# 使用 concat() 將新資料合併到舊的 df 中
df = pd.concat([df, new_data], ignore_index=True)

In [10]:
df

,日期,時間,分類,品項,金額,付款人,備註
0,2026-03-05,9:27,,早餐,50,,
1,2026-03-04,16:25,,麵包,50,ay,無
2,2026-03-03,15:04,,書,350,k,無
3,2026-02-15,15:57,食,零食,30.0,NaN,NaN


In [12]:
# 將 DataFrame 的最後一列轉換成列表的列表 (list of lists)，這是 gspread 支援的資料格式
# 替換 NaN 為 None，因為 gspread 不支援 NaN
data_to_write = df.iloc[-1:].replace({float('nan'): None}).values.tolist()

# 第一步：取得工作表物件
worksheet = gsheets.worksheet('工作表1')

# 第二步：將資料寫入工作表
# append_rows 預期接收一個列表的列表，即使只有一行資料也需要是 [['data1', 'data2']]
worksheet.append_rows(values=data_to_write, value_input_option='USER_ENTERED')
print("資料已成功寫入 Google 工作表！")

資料已成功寫入 Google 工作表！


In [13]:
data_to_write[-1]

['2026-02-15', '15:57', '食', '零食', 30.0, None, None]